### 🔍 Task 2.5: Landmark Frequency & Vocabulary Discovery
This section uses SpaCy to perform Part-of-Speech (POS) tagging on the RVS instructions. By extracting high-frequency nouns, we can ensure our config.py has the necessary "Map Vocabulary" to resolve human-described landmarks (e.g., "Bodega", "Deli", "Station") to OpenStreetMap tags.

In [4]:
import pandas as pd
import spacy
from collections import Counter
import re
import os

# 1. Load the SpaCy model (Small and fast for frequency analysis)
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading SpaCy English model...")
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

def extract_landmarks_spacy(file_path, text_column='content'):
    """Extracts nouns from navigation text to find common landmarks."""
    
    # 1. Load Data with correct format handling
    if file_path.endswith('.parquet'):
        df = pd.read_parquet(file_path)
    elif file_path.endswith('.json'):
        # 'lines=True' is the key for the RVS manhattan.json format
        df = pd.read_json(file_path, lines=True)
    else:
        df = pd.read_csv(file_path)

    print(f"Loaded {len(df)} instructions. Analyzing spatial nouns...")

    all_nouns = []
    
    # Navigation-specific stop words (words that are nouns but not landmarks)
    exclude = {
        'meters', 'feet', 'blocks', 'turn', 'left', 'right', 'straight', 
        'way', 'distance', 'steps', 'location', 'point', 'side', 'corner'
    }

    # 2. Batch Process for Speed
    # .pipe is much faster than a loop for thousands of instructions
    texts = df[text_column].dropna().astype(str).tolist()
    
    for doc in nlp.pipe(texts, disable=["ner", "parser"]):
        # Extract nouns (NN) and proper nouns (PROPN)
        nouns = [token.lemma_.lower() for token in doc 
                 if token.pos_ in {"NOUN", "PROPN"} 
                 and token.lemma_.lower() not in exclude]
        all_nouns.extend(nouns)

    # 3. Count and Report
    counts = Counter(all_nouns)
    
    print("\n--- Top 30 High-Frequency Landmarks (Manhattan) ---")
    print(f"{'WORD (Lemma)':15} | {'COUNT'}")
    print("-" * 35)
    for word, freq in counts.most_common(30):
        print(f"{word:15} | {freq}")
    
    return counts

# --- EXECUTION ---
file_path = '../data/manhattan/manhattan.json'

# Run the extraction on the 'content' column
landmark_counts = extract_landmarks_spacy(file_path, text_column='content')

# Optional: View the results as a DataFrame for easier reading
import pandas as pd
top_landmarks_df = pd.DataFrame(landmark_counts.most_common(50), columns=['Landmark', 'Frequency'])
print(top_landmarks_df.head(10))

Loaded 7000 instructions. Analyzing spatial nouns...

--- Top 30 High-Frequency Landmarks (Manhattan) ---
WORD (Lemma)    | COUNT
-----------------------------------
block           | 7316
street          | 7038
west            | 3693
east            | 2796
north           | 2014
restaurant      | 1973
parking         | 1912
shop            | 1856
avenue          | 1677
bicycle         | 1636
park            | 1234
south           | 888
bench           | 880
church          | 868
southwest       | 852
head            | 835
garden          | 683
hotel           | 660
bank            | 637
cafe            | 621
middle          | 542
food            | 540
place           | 527
pharmacy        | 506
southeast       | 491
bike            | 467
water           | 465
end             | 444
museum          | 416
rental          | 398
     Landmark  Frequency
0       block       7316
1      street       7038
2        west       3693
3        east       2796
4       north       2014
5  restaurant

Check config.py coverage:

In [ ]:
def check_vocabulary_coverage(counts, landmark_groups):
    """
    Checks how many of the top extracted nouns are covered by our config.
    """
    # Get all human-used words from the SpaCy counts
    human_words = [word for word, freq in counts.most_common(50)]
    
    # Get all words our config currently 'understands'
    config_keys = set(landmark_groups.keys())
    
    covered = [word for word in human_words if word.upper() in config_keys]
    missing = [word for word in human_words if word.upper() not in config_keys]
    
    coverage_pct = (len(covered) / len(human_words)) * 100
    
    print(f"--- Vocabulary Coverage Report ---")
    print(f"Coverage of Top 50 Nouns: {coverage_pct:.2f}%")
    print(f"\n✅ Covered: {', '.join(covered[:10])}...")
    print(f"\n❌ Missing from Config: {', '.join(missing[:15])}")
    
    return missing

#aaaaa
import sys
import os

# 1. Manually add the project root to sys.path so it can find config.py
# This assumes your structure is: project_root/notebooks/your_notebook.ipynb
# and config.py is in project_root/
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Now try the import again
try:
    from config import LANDMARK_GROUPS
    print("✅ config.py loaded successfully!")
except ImportError:
    print("❌ Still can't find config.py. Check if the file exists in:", project_root)

# 3. Run the coverage check
missing_words = check_vocabulary_coverage(landmark_counts, LANDMARK_GROUPS)

✅ config.py loaded successfully!
--- Vocabulary Coverage Report ---
Coverage of Top 50 Nouns: 32.00%

✅ Covered: street, restaurant, parking, shop, avenue, bicycle, park, bench, church, garden...

❌ Missing from Config: block, west, east, north, south, southwest, head, middle, food, place, southeast, bike, end, rental, road


In [15]:
def check_vocabulary_coverage_refined(counts, landmark_groups):
    # Words that are nouns but NOT landmarks (The noise we found)
    noise = {
        'block', 'west', 'east', 'north', 'south', 'southwest', 
        'southeast', 'head', 'middle', 'end', 'place', 'road', 
        'side', 'corner', 'distance', 'way'
    }
    
    # Filter the Top 50 to only include potential landmarks
    human_landmarks = [word for word, freq in counts.most_common(70) if word not in noise]
    # Take the top 30 actual landmark candidates
    human_landmarks = human_landmarks[:30]
    
    config_keys = set(landmark_groups.keys())
    covered = [word for word in human_landmarks if word.upper() in config_keys]
    
    coverage_pct = (len(covered) / len(human_landmarks)) * 100
    print(f"--- Refined Landmark Coverage ---")
    print(f"Coverage of Actual Landmarks: {coverage_pct:.2f}%")
    return coverage_pct


import sys
import os

# 1. Manually add the project root to sys.path so it can find config.py
# This assumes your structure is: project_root/notebooks/your_notebook.ipynb
# and config.py is in project_root/
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# 2. Now try the import again
try:
    from config import LANDMARK_GROUPS
    print("✅ config.py loaded successfully!")
except ImportError:
    print("❌ Still can't find config.py. Check if the file exists in:", project_root)

# 3. Run the coverage check
missing_words = check_vocabulary_coverage_refined(landmark_counts, LANDMARK_GROUPS)

✅ config.py loaded successfully!
--- Refined Landmark Coverage ---
Coverage of Actual Landmarks: 53.33%


In [16]:
def debug_missing_landmarks(counts, landmark_groups):
    noise = {'block', 'west', 'east', 'north', 'south', 'southwest', 'southeast', 
             'head', 'middle', 'end', 'place', 'road', 'side', 'corner', 'distance', 'way'}
    
    # Filter for potential landmarks
    human_candidates = [word for word, freq in counts.most_common(70) if word not in noise]
    human_candidates = human_candidates[:30] # Top 30 actual landmark candidates
    
    config_keys = set(landmark_groups.keys())
    
    missing = [word for word in human_candidates if word.upper() not in config_keys]
    
    print(f"--- THE HIT LIST (Top 30 candidates not in Config) ---")
    for word in missing:
        # Get frequency from original counts
        print(f"MISSING: {word:15} | Freq: {counts[word]}")
    
    return missing

# RUN THIS NOW
missing_list = debug_missing_landmarks(landmark_counts, LANDMARK_GROUPS)

--- THE HIT LIST (Top 30 candidates not in Config) ---
MISSING: food            | Freq: 540
MISSING: bike            | Freq: 467
MISSING: rental          | Freq: 398
MISSING: building        | Freq: 377
MISSING: broadway        | Freq: 373
MISSING: northwest       | Freq: 349
MISSING: post            | Freq: 346
MISSING: northeast       | Freq: 345
MISSING: bar             | Freq: 338
MISSING: school          | Freq: 331
MISSING: library         | Freq: 326
MISSING: station         | Freq: 304
MISSING: store           | Freq: 297
MISSING: office          | Freq: 290


In [ ]:
def check_vocabulary_coverage_refined2(counts, landmark_groups):
    # Words that are nouns but NOT landmarks (The noise we found)
    noise = {
        'block', 'west', 'east', 'north', 'south', 'southwest', 'southeast', 
        'northwest', 'northeast', 'head', 'middle', 'end', 'place', 'road', 
        'side', 'corner', 'distance', 'way'
    }
    
    # Filter the Top 50 to only include potential landmarks
    human_landmarks = [word for word, freq in counts.most_common(70) if word not in noise]
    # Take the top 30 actual landmark candidates
    human_landmarks = human_landmarks[:30]
    
    config_keys = set(landmark_groups.keys())
    covered = [word for word in human_landmarks if word.upper() in config_keys]
    
    coverage_pct = (len(covered) / len(human_landmarks)) * 100
    print(f"--- Refined Landmark Coverage ---")
    print(f"Coverage of Actual Landmarks: {coverage_pct:.2f}%")
    return coverage_pct

import sys
import os
import importlib  # <--- New import, needed for reloading config module after edits

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# 1. Import or RELOAD the config
import config
importlib.reload(config) # <--- This forces Python to look at the disk
from config import LANDMARK_GROUPS

print(f"✅ config.py reloaded! Current groups: {len(LANDMARK_GROUPS)}")

# 2. Run the refined coverage check
coverage_pct = check_vocabulary_coverage_refined2(landmark_counts, LANDMARK_GROUPS)

# 3. Print the missing ones again to see if we missed any capitalization issues
noise = {'block', 'west', 'east', 'north', 'south', 'southwest', 'southeast', 
         'northwest', 'northeast', 'head', 'middle', 'end', 'place', 'road', 
         'side', 'corner', 'distance', 'way'}
human_landmarks = [word for word, freq in landmark_counts.most_common(70) if word not in noise][:30]
missing = [word for word in human_landmarks if word.upper() not in LANDMARK_GROUPS.keys()]
print(f"Remaining Missing: {missing}")

✅ config.py reloaded! Current groups: 28
--- Refined Landmark Coverage ---
Coverage of Actual Landmarks: 93.33%
Remaining Missing: ['hotel', 'intersection']
